# Notebook Goal

This notebook builds and tests a reusable inference pipeline for the Credit Card Fraud Detection project.

Notebook `17_final_model_training.ipynb` already trained and saved the final validated fraud model and its supporting artifacts. Notebook `18_inference_pipeline.ipynb` will load those saved artifacts and use them to make predictions.

No model training happens in this notebook. No threshold tuning happens in this notebook. The selected features and saved decision policy from notebook 17 are reused as-is.

The purpose of this notebook is to simulate how the fraud model will work in production: load the saved model, prepare input data in the expected format, generate a fraud probability, and convert that result into a reusable prediction workflow.

This notebook also prepares the project for the next FastAPI `/predict` endpoint by turning the saved model artifacts into a clear, testable inference path.

In this reusable-helper step, the core inference logic is moved into `src/inference/fraud_inference.py`. The notebook now imports those helpers and focuses on smoke tests only. No training, threshold tuning, or artifact modification happens in this step.


# Load Saved Artifacts

This section loads the final validated model and the supporting JSON artifacts created in notebook `17_final_model_training.ipynb`.

The goal here is only to verify that the saved inference assets exist and can be loaded correctly before later steps use them for prediction.

The reusable helper module reads the same saved artifacts, so this notebook checks the shared inference path that a future API will also use.


In [118]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier


In [119]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.inference.fraud_inference import (
    apply_decision_policy,
    load_artifacts,
    predict_batch,
    predict_fraud,
    prepare_model_input,
    validate_transaction_input,
)


In [120]:
artifacts = load_artifacts(project_root=PROJECT_ROOT)
artifact_paths = artifacts["artifact_paths"]

for artifact_name, artifact_path in artifact_paths.items():
    print(f"Found {artifact_name}: {artifact_path}")

final_model = artifacts["model"]
final_feature_columns = artifacts["feature_columns"]
final_decision_policy = artifacts["decision_policy"]
final_model_metadata = artifacts["model_metadata"]
final_model_metrics = artifacts["model_metrics"]

print(f"Loaded final validated model successfully: {artifact_paths['final_validated_model']}")
print(f"Loaded final feature columns successfully: {artifact_paths['final_feature_columns']}")
print(f"Loaded final decision policy successfully: {artifact_paths['final_decision_policy']}")
print(f"Loaded final model metadata successfully: {artifact_paths['final_model_metadata']}")
print(f"Loaded final model metrics successfully: {artifact_paths['final_model_metrics']}")


Found final_validated_model: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_validated_fraud_model.joblib
Found final_feature_columns: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_feature_columns.json
Found final_decision_policy: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_decision_policy.json
Found final_model_metadata: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_model_metadata.json
Found final_model_metrics: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_model_metrics.json
Loaded final validated model successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_validated_fraud_model.joblib
Loaded final feature columns successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_feature_columns.json
Loaded final decision policy successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Det

# Validate Artifact Consistency

This section checks that the loaded model and saved artifacts are aligned before they are used for inference.

If anything important is missing or mismatched, the notebook stops early with a clear validation error.


In [121]:
if not isinstance(final_feature_columns, list) or not final_feature_columns:
    raise ValueError("Artifact validation failed: final_feature_columns must be a non-empty list.")

if not isinstance(final_model, RandomForestClassifier):
    raise ValueError(
        "Artifact validation failed: loaded model must be a RandomForestClassifier. "
        f"Found {type(final_model).__name__}."
    )

if "review_threshold" not in final_decision_policy:
    raise ValueError(
        "Artifact validation failed: final_decision_policy is missing review_threshold."
    )

if "block_threshold" not in final_decision_policy:
    raise ValueError(
        "Artifact validation failed: final_decision_policy is missing block_threshold."
    )

review_threshold = final_decision_policy["review_threshold"]
block_threshold = final_decision_policy["block_threshold"]

if review_threshold >= block_threshold:
    raise ValueError(
        "Artifact validation failed: review_threshold must be less than block_threshold. "
        f"Found review_threshold={review_threshold} and block_threshold={block_threshold}."
    )

actual_feature_count = len(final_feature_columns)

if hasattr(final_model, "n_features_in_"):
    expected_feature_count = final_model.n_features_in_
elif "feature_count" in final_model_metadata:
    expected_feature_count = final_model_metadata["feature_count"]
else:
    raise ValueError(
        "Artifact validation failed: could not determine the model's expected feature count."
    )

if expected_feature_count != actual_feature_count:
    raise ValueError(
        "Artifact validation failed: model feature count does not match final_feature_columns. "
        f"Expected {expected_feature_count}, found {actual_feature_count}."
    )

model_version = final_model_metadata.get("model_version") or final_model_metadata.get("version")

print(f"Model type: {type(final_model).__name__}")
print(f"Expected feature count: {expected_feature_count}")
print(f"Actual feature count: {actual_feature_count}")
print(f"Review threshold: {review_threshold}")
print(f"Block threshold: {block_threshold}")
print(f"Model version: {model_version if model_version is not None else 'Not available'}")


Model type: RandomForestClassifier
Expected feature count: 13
Actual feature count: 13
Review threshold: 0.35
Block threshold: 0.5
Model version: Not available


# Use Input Validation Helper

This section uses the reusable input validation helper from `src/inference/fraud_inference.py`.

The imported helper checks required features, verifies numeric values, and safely ignores extra input columns. The notebook now reuses this shared logic instead of redefining it locally.


In [122]:
print("Imported validate_transaction_input from src/inference/fraud_inference.py")


Imported validate_transaction_input from src/inference/fraud_inference.py


# Use Preprocessing Helper

This section uses the reusable preprocessing helper to prepare one validated transaction in the exact feature order expected by the saved model.

The helper always follows the saved `final_feature_columns.json` order so later prediction steps never depend on random input ordering.


In [123]:
print("Imported prepare_model_input from src/inference/fraud_inference.py")


Imported prepare_model_input from src/inference/fraud_inference.py


In [124]:
processed_sample_path = PROJECT_ROOT / "data" / "processed" / "final_features.csv"

if processed_sample_path.exists():
    sample_source_df = pd.read_csv(processed_sample_path, nrows=1)
    sample_transaction = sample_source_df.drop(columns=["Class"], errors="ignore").iloc[0].to_dict()
    sample_source = f"processed sample from {processed_sample_path.name}"
else:
    sample_transaction = {
        feature_name: float(index)
        for index, feature_name in enumerate(final_feature_columns, start=1)
    }
    sample_source = "dummy sample built from final_feature_columns"

sample_transaction["extra_input_column"] = 999.0

prepared_sample_input = prepare_model_input(sample_transaction, final_feature_columns)

assert prepared_sample_input.shape == (1, len(final_feature_columns))
assert prepared_sample_input.columns.tolist() == final_feature_columns
assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in prepared_sample_input.dtypes)

print(f"Prepared sample source: {sample_source}")
print(prepared_sample_input.shape)
print(prepared_sample_input.columns.tolist())
prepared_sample_input.head()


Prepared sample source: processed sample from final_features.csv
(1, 13)
['V14_V12_interaction', 'V14', 'V17_V16_interaction', 'V12', 'V17', 'V10', 'V4', 'V16', 'V3', 'V11', 'V7', 'V18', 'log_amount']


,V14_V12_interaction,V14,V17_V16_interaction,V12,V17,V10,V4,V16,V3,V11,V7,V18,log_amount
0,0.192241,-0.311169,-0.09783,-0.617801,0.207971,0.090794,1.378155,-0.470401,2.536347,-0.5516,0.239599,0.025791,5.01476


# Use Prediction Helpers

This section uses the reusable prediction helpers from `src/inference/fraud_inference.py`.

`apply_decision_policy`, `predict_fraud`, and `predict_batch` all reuse the saved artifacts without retraining the model. The notebook only smoke-tests the imported helpers.


In [125]:
print("Imported apply_decision_policy, predict_fraud, and predict_batch from src/inference/fraud_inference.py")


Imported apply_decision_policy, predict_fraud, and predict_batch from src/inference/fraud_inference.py


In [126]:
sample_prediction = predict_fraud(
    sample_transaction,
    model=final_model,
    feature_columns=final_feature_columns,
    decision_policy=final_decision_policy,
    model_metadata=final_model_metadata,
)
sample_prediction


{'fraud_probability': 0.0,
 'decision': 'APPROVE',
 'risk_level': 'LOW',
 'reason': 'Transaction probability is below review threshold.',
 'thresholds': {'review_threshold': 0.35, 'block_threshold': 0.5},
 'model_version': 'Not available'}

In [127]:
block_test   = final_decision_policy["block_threshold"]
review_test  = final_decision_policy["review_threshold"]
approve_test = round(final_decision_policy["review_threshold"] - 0.01, 4)

block_result   = apply_decision_policy(block_test,   final_decision_policy)
review_result  = apply_decision_policy(review_test,  final_decision_policy)
approve_result = apply_decision_policy(approve_test, final_decision_policy)

assert block_result["decision"]   == "BLOCK"  and block_result["risk_level"]   == "HIGH"
assert review_result["decision"]  == "REVIEW" and review_result["risk_level"]  == "MEDIUM"
assert approve_result["decision"] == "APPROVE" and approve_result["risk_level"] == "LOW"

print(f"BLOCK   (p={block_test}):   {block_result}")
print(f"REVIEW  (p={review_test}):  {review_result}")
print(f"APPROVE (p={approve_test}): {approve_result}")


BLOCK   (p=0.5):   {'decision': 'BLOCK', 'risk_level': 'HIGH', 'reason': 'Transaction probability is above block threshold.'}
REVIEW  (p=0.35):  {'decision': 'REVIEW', 'risk_level': 'MEDIUM', 'reason': 'Transaction probability is between review and block thresholds.'}
APPROVE (p=0.34): {'decision': 'APPROVE', 'risk_level': 'LOW', 'reason': 'Transaction probability is below review threshold.'}


# Test with Sample Transactions

This section uses a few sample transactions only to prove that the saved inference pipeline works end to end.

It does not retrain the model, tune thresholds, or recalculate overall performance metrics.

In [128]:
phase8_dataset_path = PROJECT_ROOT / "data" / "processed" / "final_features.csv"
phase8_samples = []

if phase8_dataset_path.exists():
    phase8_df = pd.read_csv(phase8_dataset_path)

    normal_rows = phase8_df[phase8_df["Class"] == 0]
    fraud_rows = phase8_df[phase8_df["Class"] == 1]

    if not normal_rows.empty:
        normal_transaction = normal_rows.iloc[0][final_feature_columns].to_dict()
        phase8_samples.append(
            {
                "sample_type": "Actual normal transaction",
                "actual_class": int(normal_rows.iloc[0]["Class"]),
                "transaction": normal_transaction,
            }
        )

    if not fraud_rows.empty:
        fraud_transaction = fraud_rows.iloc[0][final_feature_columns].to_dict()
        phase8_samples.append(
            {
                "sample_type": "Actual fraud transaction",
                "actual_class": int(fraud_rows.iloc[0]["Class"]),
                "transaction": fraud_transaction,
            }
        )

    if not normal_rows.empty and not fraud_rows.empty:
        normal_series = normal_rows.iloc[0][final_feature_columns]
        fraud_series = fraud_rows.iloc[0][final_feature_columns]
        borderline_transaction = ((normal_series + fraud_series) / 2.0).to_dict()
        phase8_samples.append(
            {
                "sample_type": "Manually created borderline transaction",
                "actual_class": "Not available",
                "transaction": borderline_transaction,
            }
        )

if not phase8_samples:
    dummy_base = {
        feature_name: float(index)
        for index, feature_name in enumerate(final_feature_columns, start=1)
    }

    phase8_samples = [
        {
            "sample_type": "Dummy smoke test 1",
            "actual_class": "Not available",
            "transaction": dummy_base,
        },
        {
            "sample_type": "Dummy smoke test 2",
            "actual_class": "Not available",
            "transaction": {
                feature_name: value + 0.5 for feature_name, value in dummy_base.items()
            },
        },
        {
            "sample_type": "Dummy smoke test 3",
            "actual_class": "Not available",
            "transaction": {
                feature_name: value - 0.5 for feature_name, value in dummy_base.items()
            },
        },
    ]

for sample in phase8_samples:
    prediction = predict_fraud(
        sample["transaction"],
        model=final_model,
        feature_columns=final_feature_columns,
        decision_policy=final_decision_policy,
        model_metadata=final_model_metadata,
    )
    print(f"Sample type: {sample['sample_type']}")
    print(f"Actual class: {sample['actual_class']}")
    print(f"Fraud probability: {prediction['fraud_probability']:.6f}")
    print(f"Decision: {prediction['decision']}")
    print(f"Risk level: {prediction['risk_level']}")
    print(f"Reason: {prediction['reason']}")
    print("-" * 60)


Sample type: Actual normal transaction
Actual class: 0
Fraud probability: 0.000000
Decision: APPROVE
Risk level: LOW
Reason: Transaction probability is below review threshold.
------------------------------------------------------------
Sample type: Actual fraud transaction
Actual class: 1
Fraud probability: 0.913333
Decision: BLOCK
Risk level: HIGH
Reason: Transaction probability is above block threshold.
------------------------------------------------------------
Sample type: Manually created borderline transaction
Actual class: Not available
Fraud probability: 0.056667
Decision: APPROVE
Risk level: LOW
Reason: Transaction probability is below review threshold.
------------------------------------------------------------


# Batch Prediction Test

This section uses the reusable batch prediction helper for production-style testing with multiple rows at once.

It uses the saved model and saved decision policy only, without retraining or recalculating evaluation metrics.


In [129]:
processed_dataset_path = PROJECT_ROOT / "data" / "processed" / "final_features.csv"

if processed_dataset_path.exists():
    sample_source_df = pd.read_csv(processed_dataset_path).head(6).copy()
    sample_transactions_df = sample_source_df[final_feature_columns].copy()
    sample_transactions_df.insert(0, "sample_id", [f"sample_{i}" for i in range(len(sample_transactions_df))])
    sample_actual_classes = sample_source_df["Class"].reset_index(drop=True)
else:
    dummy_transaction_rows = []
    for row_number in range(5):
        dummy_transaction_rows.append(
            {
                "sample_id": f"dummy_{row_number}",
                **{
                    feature_name: float(index + row_number)
                    for index, feature_name in enumerate(final_feature_columns, start=1)
                },
            }
        )

    sample_transactions_df = pd.DataFrame(dummy_transaction_rows)
    sample_actual_classes = pd.Series(["Not available"] * len(sample_transactions_df))

batch_predictions_df = predict_batch(
    sample_transactions_df,
    model=final_model,
    feature_columns=final_feature_columns,
    decision_policy=final_decision_policy,
)
batch_results_display_df = batch_predictions_df.copy()
batch_results_display_df.insert(1, "actual_class", sample_actual_classes.values)
batch_results_display_df


,sample_id,actual_class,fraud_probability,decision,risk_level,reason
0,sample_0,0,0.0,APPROVE,LOW,Transaction probability is below review thresh...
1,sample_1,0,0.0,APPROVE,LOW,Transaction probability is below review thresh...
2,sample_2,0,0.0,APPROVE,LOW,Transaction probability is below review thresh...
3,sample_3,0,0.0,APPROVE,LOW,Transaction probability is below review thresh...
4,sample_4,0,0.0,APPROVE,LOW,Transaction probability is below review thresh...
5,sample_5,0,0.0,APPROVE,LOW,Transaction probability is below review thresh...


# Error Handling Tests

This section checks that the inference helpers fail with clear messages for common bad inputs.

These are smoke tests for input handling only. They do not retrain the model or evaluate model performance.


In [130]:
error_test_transaction = {
    feature_name: float(index)
    for index, feature_name in enumerate(final_feature_columns, start=1)
}

error_test_cases = [
    (
        "Missing required feature",
        lambda: predict_fraud(
            {
                key: value
                for key, value in error_test_transaction.items()
                if key != final_feature_columns[0]
            },
            model=final_model,
            feature_columns=final_feature_columns,
            decision_policy=final_decision_policy,
            model_metadata=final_model_metadata,
        ),
    ),
    (
        "Non-numeric feature value",
        lambda: predict_fraud(
            {
                **error_test_transaction,
                final_feature_columns[0]: "not_numeric",
            },
            model=final_model,
            feature_columns=final_feature_columns,
            decision_policy=final_decision_policy,
            model_metadata=final_model_metadata,
        ),
    ),
    (
        "Empty input dictionary",
        lambda: predict_fraud(
            {},
            model=final_model,
            feature_columns=final_feature_columns,
            decision_policy=final_decision_policy,
            model_metadata=final_model_metadata,
        ),
    ),
    (
        "Empty batch DataFrame",
        lambda: predict_batch(pd.DataFrame(columns=final_feature_columns)),
    ),
]

for test_name, test_function in error_test_cases:
    print(f"Test: {test_name}")
    try:
        test_function()
        print("Unexpected result: no error was raised.")
    except ValueError as error:
        print(f"Caught ValueError: {error}")
    print("-" * 60)

print("Test: Extra columns in single prediction")
extra_column_transaction = {
    **error_test_transaction,
    "extra_input_column": 999.0,
}

try:
    extra_column_prediction = predict_fraud(
        extra_column_transaction,
        model=final_model,
        feature_columns=final_feature_columns,
        decision_policy=final_decision_policy,
        model_metadata=final_model_metadata,
    )
    print("Extra columns were handled without crashing.")
    print(extra_column_prediction)
except ValueError as error:
    print(f"Unexpected ValueError: {error}")
print("-" * 60)

print("Test: Extra columns in batch prediction")
extra_column_batch_df = sample_transactions_df.head(2).copy()
extra_column_batch_df["extra_input_column"] = [111.0, 222.0]

try:
    extra_column_batch_predictions = predict_batch(
        extra_column_batch_df,
        model=final_model,
        feature_columns=final_feature_columns,
        decision_policy=final_decision_policy,
    )
    print("Extra batch columns were handled without crashing.")
    extra_column_batch_predictions
except ValueError as error:
    print(f"Unexpected ValueError: {error}")


Test: Missing required feature
Caught ValueError: Transaction input validation failed: missing required features: ['V14_V12_interaction']
------------------------------------------------------------
Test: Non-numeric feature value
Caught ValueError: Transaction input validation failed: required features must be numeric. Found invalid values: ["V14_V12_interaction='not_numeric' (str)"]
------------------------------------------------------------
Test: Empty input dictionary
Caught ValueError: Transaction input validation failed: transaction input cannot be empty.
------------------------------------------------------------
Test: Empty batch DataFrame
Caught ValueError: Batch prediction failed: input DataFrame cannot be empty.
------------------------------------------------------------
Test: Extra columns in single prediction
Extra columns were handled without crashing.
{'fraud_probability': 0.013333333333333334, 'decision': 'APPROVE', 'risk_level': 'LOW', 'reason': 'Transaction probabi

# Reusable Inference Helper Module

The core inference logic has been moved into `src/inference/fraud_inference.py`.

The notebook now imports `load_artifacts`, `validate_transaction_input`, `prepare_model_input`, `apply_decision_policy`, `predict_fraud`, and `predict_batch` from that module instead of redefining them locally.

The helper module uses the same saved artifacts from `17_final_model_training.ipynb` and is ready to be imported by the FastAPI endpoint in the next step.